In [3]:
# INDICATORE UTILIZZAZIONE LORDA 

from pathlib import Path
import pandas as pd
import duckdb

BASE_DIR = Path(r"../")
RAW_DIR = BASE_DIR / "data" / "raw"

def chiave_comune(s):
    if pd.isna(s):
        return None
    apostrofo_tipografico = chr(0x2019)
    return str(s).strip().lower().replace(apostrofo_tipografico, "'")

ANNI = [2022, 2023, 2024, 2025]

FILE_PORTI_AEROPORTI = RAW_DIR / "porti_aeroporti.csv"
print("CSV utilizzati:")
print(f"  {FILE_PORTI_AEROPORTI}")

pa = pd.read_csv(FILE_PORTI_AEROPORTI)
pa["data"] = pd.to_datetime(pa["data"])
pa["anno"] = pa["data"].dt.year
pa["mese"] = pa["data"].dt.month
arrivi_mensili_pa = pa.groupby(["anno", "mese"], as_index=False)["arrivi"].sum()
arrivi_annuali_pa = arrivi_mensili_pa.groupby("anno")["arrivi"].transform("sum")
arrivi_mensili_pa["peso_mese"] = (arrivi_mensili_pa["arrivi"] / arrivi_annuali_pa).round(6)


CSV utilizzati:
  ../data/raw/porti_aeroporti.csv


In [4]:
# presenze annuali per comune E per macro-tipologia (serve per isolare i segmenti anomali)
presenze_annue_macro = {}
for anno in ANNI:
    FILE_PRESENZE = RAW_DIR / f"csv_opendata_comuni_{anno}.csv"
    print(f"\n=== FILE: {FILE_PRESENZE.name} ===")

    # il nome della colonna macro-tipologia varia tra gli anni (trattino/underscore)
    intestazione = pd.read_csv(FILE_PRESENZE, nrows=1)
    col_macro = "macro-tipologia" if "macro-tipologia" in intestazione.columns else "macro_tipologia"

    df = pd.read_csv(FILE_PRESENZE, usecols=["comune", "mese", "presenze", col_macro])
    df = df.rename(columns={col_macro: "macro_tipologia_raw"})

    mask_comune_ignoto = df["comune"].astype(str).str.strip().str.lower() == "non disponibile"
    df = df[~mask_comune_ignoto].copy()
    df["comune"] = df["comune"].astype(str).str.strip().str.title()
    df["chiave_comune"] = df["comune"].map(chiave_comune)
    df["macro_tipologia"] = df["macro_tipologia_raw"].replace("non disponibile", "Non Classificato")

    mask_non_disp = df["mese"].astype(str).str.strip().str.lower() == "non disponibile"
    non_allocabili = (
        df[mask_non_disp].groupby(["chiave_comune", "macro_tipologia"], as_index=False)["presenze"]
        .sum().rename(columns={"presenze": "presenze_non_allocabili"})
    )

    df_mensile = df[~mask_non_disp].copy()
    df_mensile["mese"] = df_mensile["mese"].astype(int)
    agg = (
        df_mensile.groupby(["chiave_comune", "macro_tipologia", "mese"], as_index=False)["presenze"]
        .sum().rename(columns={"presenze": "presenze_mese"})
    )

    pesi_anno = arrivi_mensili_pa[arrivi_mensili_pa["anno"] == anno][["mese", "peso_mese"]]
    redistrib = non_allocabili.merge(pesi_anno, how="cross")
    redistrib["presenze_redistribuite"] = (
        redistrib["presenze_non_allocabili"] * redistrib["peso_mese"]
    ).round(2)
    redistrib = redistrib[["chiave_comune", "macro_tipologia", "mese", "presenze_redistribuite"]]

    m = agg.merge(redistrib, on=["chiave_comune", "macro_tipologia", "mese"], how="outer")
    m["presenze_mese"] = m["presenze_mese"].fillna(0)
    m["presenze_redistribuite"] = m["presenze_redistribuite"].fillna(0)
    m["presenze_mese"] = (m["presenze_mese"] + m["presenze_redistribuite"]).round(2)

    ann_macro = (
        m.groupby(["chiave_comune", "macro_tipologia"], as_index=False)["presenze_mese"]
        .sum().rename(columns={"presenze_mese": "presenze_annue"})
    )
    comuni_map = df[["chiave_comune", "comune"]].drop_duplicates()
    ann_macro = ann_macro.merge(comuni_map, on="chiave_comune", how="left")
    ann_macro["anno"] = anno
    presenze_annue_macro[anno] = ann_macro[["chiave_comune", "comune", "anno", "macro_tipologia", "presenze_annue"]]
    print(f"{anno}: colonna letta come '{col_macro}' | {ann_macro['comune'].nunique()} comuni")


=== FILE: csv_opendata_comuni_2022.csv ===
2022: colonna letta come 'macro_tipologia' | 189 comuni

=== FILE: csv_opendata_comuni_2023.csv ===
2023: colonna letta come 'macro-tipologia' | 288 comuni

=== FILE: csv_opendata_comuni_2024.csv ===
2024: colonna letta come 'macro-tipologia' | 307 comuni

=== FILE: csv_opendata_comuni_2025.csv ===
2025: colonna letta come 'macro-tipologia' | 322 comuni


In [5]:
print("\n--- Verifica Cagliari 2025, per macro-tipologia ---")
print(presenze_annue_macro[2025][presenze_annue_macro[2025]["comune"] == "Cagliari"].to_string(index=False))


--- Verifica Cagliari 2025, per macro-tipologia ---
chiave_comune   comune  anno                                       macro_tipologia  presenze_annue
     cagliari Cagliari  2025                                  Esercizi Alberghieri        444312.0
     cagliari Cagliari  2025 Esercizi Extra-Alberghieri:Alloggi privati in affitto        390981.0
     cagliari Cagliari  2025     Esercizi Extra-Alberghieri:Esercizi Complementari        288142.0


In [6]:
def classifica_tipologia(tip):
    """Mappatura best-effort tipologia struttura -> macro-categoria (validata: 0 letti persi)."""
    t = str(tip).lower()
    if any(k in t for k in ["albergo", "residence", "r.t.a", "marina resort", "villaggio albergo"]):
        return "Esercizi Alberghieri"
    if any(k in t for k in ["alloggi privati", "casa/app", "case e appartamenti", "case per vacanze"]):
        return "Esercizi Extra-Alberghieri:Alloggi privati in affitto"
    return "Esercizi Extra-Alberghieri:Esercizi Complementari"

In [7]:
FILE_SUPERFICIE = RAW_DIR / "superficie_comunale.csv"
superficie = pd.read_csv(FILE_SUPERFICIE)
superficie["comune"] = superficie["comune"].astype(str).str.strip().str.title()
superficie["chiave_comune"] = superficie["comune"].map(chiave_comune)
print(f"=== FILE: {FILE_SUPERFICIE.name} ===")
print(f"comuni: {superficie['comune'].nunique()}")

=== FILE: superficie_comunale.csv ===
comuni: 377


In [8]:
FILE_CAPACITA = RAW_DIR / "capacita_ricettiva.csv"
cap = pd.read_csv(FILE_CAPACITA)
cap["comune"] = cap["comune"].astype(str).str.strip().str.title()
cap["chiave_comune"] = cap["comune"].map(chiave_comune)
cap["macro_tipologia"] = cap["tipologia"].map(classifica_tipologia)
print(f"\n=== FILE: {FILE_CAPACITA.name} ===")
print(f"righe sorgente: {len(cap)}")


=== FILE: capacita_ricettiva.csv ===
righe sorgente: 8589


In [9]:
letti_annuali_macro = (
    cap.groupby(["chiave_comune", "anno", "macro_tipologia"], as_index=False)["letti"]
    .sum().rename(columns={"letti": "letti_totali"})
)

print("\n--- Verifica Cagliari 2025, letti per macro-tipologia ---")
print(letti_annuali_macro[(letti_annuali_macro["chiave_comune"] == "cagliari") &
                           (letti_annuali_macro["anno"] == 2025)].to_string(index=False))


--- Verifica Cagliari 2025, letti per macro-tipologia ---
chiave_comune  anno                                       macro_tipologia  letti_totali
     cagliari  2025                                  Esercizi Alberghieri          2844
     cagliari  2025 Esercizi Extra-Alberghieri:Alloggi privati in affitto          9950
     cagliari  2025     Esercizi Extra-Alberghieri:Esercizi Complementari          4421


In [10]:
# --- unione presenze e letti per segmento, calcolo utilizzazione per segmento ---
tabelle_anno = {}
for anno in ANNI:
    seg = presenze_annue_macro[anno].merge(
        letti_annuali_macro[letti_annuali_macro["anno"] == anno],
        on=["chiave_comune", "macro_tipologia"], how="outer"
    )
    seg["presenze_annue"] = seg["presenze_annue"].fillna(0)
    seg["letti_totali"] = seg["letti_totali"].fillna(0)

    # FIX: il nome comune viene SEMPRE dall'anagrafica canonica (superficie),
    # mai da un fillna con la chiave grezza minuscola (bug che raddoppiava i comuni)
    seg = seg.drop(columns=["comune"], errors="ignore")
    seg = seg.merge(superficie[["chiave_comune", "comune"]], on="chiave_comune", how="left")

    seg["utilizzazione_segmento"] = None
    mask_seg_ok = seg["letti_totali"] > 0
    seg.loc[mask_seg_ok, "utilizzazione_segmento"] = (
        seg.loc[mask_seg_ok, "presenze_annue"] / (seg.loc[mask_seg_ok, "letti_totali"] * 365) * 100
    ).round(2)

    # oltre il 100% e' matematicamente impossibile con dati corretti -> segmento inaffidabile
    seg["segmento_anomalo"] = False
    seg.loc[mask_seg_ok, "segmento_anomalo"] = seg.loc[mask_seg_ok, "utilizzazione_segmento"] > 100

    # ricalcolo a livello comune usando SOLO i segmenti sani
    puliti = seg[~seg["segmento_anomalo"]]
    comune_level = puliti.groupby("chiave_comune", as_index=False).agg(
        presenze_annue=("presenze_annue", "sum"),
        letti_totali=("letti_totali", "sum"),
    )
    n_segmenti_esclusi = seg.groupby("chiave_comune")["segmento_anomalo"].sum().rename("segmenti_esclusi")
    comune_level = comune_level.merge(n_segmenti_esclusi, on="chiave_comune", how="left")
    comune_level = comune_level.merge(superficie[["chiave_comune", "comune"]], on="chiave_comune", how="left")

    comune_level["copertura_sufficiente"] = comune_level["letti_totali"] > 0
    comune_level["utilizzazione_lorda_pct"] = None
    mask_ok = comune_level["letti_totali"] > 0
    comune_level.loc[mask_ok, "utilizzazione_lorda_pct"] = (
        comune_level.loc[mask_ok, "presenze_annue"] /
        (comune_level.loc[mask_ok, "letti_totali"] * 365) * 100
    ).round(2)
    comune_level["anno"] = anno

    out = comune_level[["comune", "anno", "presenze_annue", "letti_totali",
                         "utilizzazione_lorda_pct", "segmenti_esclusi",
                         "copertura_sufficiente"]].sort_values("comune").reset_index(drop=True)
    tabelle_anno[anno] = out

    n_con_esclusioni = (out["segmenti_esclusi"] > 0).sum()
    print(f"{anno}: {out['comune'].nunique()} comuni | {n_con_esclusioni} comuni con almeno un segmento escluso")
    if n_con_esclusioni:
        print(out[out["segmenti_esclusi"] > 0][["comune", "segmenti_esclusi", "utilizzazione_lorda_pct"]].to_string(index=False))

2022: 329 comuni | 0 comuni con almeno un segmento escluso
2023: 334 comuni | 4 comuni con almeno un segmento escluso
                    comune  segmenti_esclusi utilizzazione_lorda_pct
                   Cardedu                 1                   21.64
                   Macomer                 1                    30.7
Trinità D'Agultu E Vignola                 1                    4.67
                Villaputzu                 1                    4.35
2024: 340 comuni | 3 comuni con almeno un segmento escluso
                    comune  segmenti_esclusi utilizzazione_lorda_pct
                   Cardedu                 1                   23.17
                     Tergu                 1                    1.01
Trinità D'Agultu E Vignola                 1                   10.97
2025: 349 comuni | 4 comuni con almeno un segmento escluso
                    comune  segmenti_esclusi utilizzazione_lorda_pct
                   Cardedu                 1                   21.41
     

In [11]:
for comune_verifica in ["Cagliari", "Uta"]:
    print(f"\n--- Verifica: '{comune_verifica}' (tutti gli anni) ---")
    r = pd.concat([tabelle_anno[a][tabelle_anno[a]["comune"] == comune_verifica] for a in ANNI])
    print(r.to_string(index=False))


--- Verifica: 'Cagliari' (tutti gli anni) ---
  comune  anno  presenze_annue  letti_totali utilizzazione_lorda_pct  segmenti_esclusi  copertura_sufficiente
Cagliari  2022        771330.0       10516.0                    20.1                 0                   True
Cagliari  2023        864079.0       11038.0                   21.45                 0                   True
Cagliari  2024        964380.0       14712.0                   17.96                 0                   True
Cagliari  2025       1123435.0       17215.0                   17.88                 0                   True

--- Verifica: 'Uta' (tutti gli anni) ---
comune  anno  presenze_annue  letti_totali utilizzazione_lorda_pct  segmenti_esclusi  copertura_sufficiente
   Uta  2022         2098.01          80.0                    7.18                 0                   True
   Uta  2023         1228.01          64.0                    5.26                 0                   True
   Uta  2024         3542.00         

In [12]:
# ============================================================
# PASSO 4 - Spalmo sui 12 mesi (valore annuale costante, coerente
# con la Densita' Ricettiva), salvataggio CSV, scrittura database
# ============================================================
OUT_DIR = BASE_DIR / "data" / "indicatore_utilizzazione_lorda"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DB_DIR = BASE_DIR / "db"
DB_PATH = DB_DIR / "sardegna_overtourism.duckdb"
DB_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(DB_PATH))
con.execute("CREATE SCHEMA IF NOT EXISTS presentation")

MESI_DF = pd.DataFrame({"mese": range(1, 13)})
tabelle_mensili = {}

for anno in ANNI:
    m_mensile = tabelle_anno[anno].merge(MESI_DF, how="cross")
    out_mensile = (
        m_mensile[["comune", "anno", "mese", "presenze_annue", "letti_totali",
                   "utilizzazione_lorda_pct", "segmenti_esclusi", "copertura_sufficiente"]]
        .sort_values(["comune", "mese"])
        .reset_index(drop=True)
    )
    tabelle_mensili[anno] = out_mensile

    percorso_out = OUT_DIR / f"utilizzazione_lorda_{anno}.csv"
    out_mensile.to_csv(percorso_out, index=False, encoding="utf-8-sig")
    print(f"{anno}: {out_mensile['comune'].nunique()} comuni | {len(out_mensile)} righe | salvato -> {percorso_out}")

#NUOVO - inserito df e cvs con i dati di tutti gli anni

df_complessivo_annuo = pd.concat(tabelle_anno.values(),ignore_index=True)
df_complessivo_annuo.to_csv(f"{OUT_DIR}/utilizzazione_lorda_complessiva_annua.csv")

df_complessivo_mensile = pd.concat(tabelle_mensili.values(),ignore_index=True)
df_complessivo_mensile.to_csv(f"{OUT_DIR}/utilizzazione_lorda_complessiva_mensile.csv")

2022: 329 comuni | 3948 righe | salvato -> ../data/indicatore_utilizzazione_lorda/utilizzazione_lorda_2022.csv
2023: 334 comuni | 4008 righe | salvato -> ../data/indicatore_utilizzazione_lorda/utilizzazione_lorda_2023.csv
2024: 340 comuni | 4080 righe | salvato -> ../data/indicatore_utilizzazione_lorda/utilizzazione_lorda_2024.csv
2025: 349 comuni | 4188 righe | salvato -> ../data/indicatore_utilizzazione_lorda/utilizzazione_lorda_2025.csv


In [13]:
tabelle_mensili

{2022:          comune  anno  mese  presenze_annue  letti_totali  \
 0     Abbasanta  2022     1         7385.01         139.0   
 1     Abbasanta  2022     2         7385.01         139.0   
 2     Abbasanta  2022     3         7385.01         139.0   
 3     Abbasanta  2022     4         7385.01         139.0   
 4     Abbasanta  2022     5         7385.01         139.0   
 ...         ...   ...   ...             ...           ...   
 3943   Zerfaliu  2022     8            0.00          23.0   
 3944   Zerfaliu  2022     9            0.00          23.0   
 3945   Zerfaliu  2022    10            0.00          23.0   
 3946   Zerfaliu  2022    11            0.00          23.0   
 3947   Zerfaliu  2022    12            0.00          23.0   
 
      utilizzazione_lorda_pct  segmenti_esclusi  copertura_sufficiente  
 0                      14.56                 0                   True  
 1                      14.56                 0                   True  
 2                      14.56

In [14]:
utilizzazione_completa = pd.concat(tabelle_mensili.values(), ignore_index=True)
con.register("utilizzazione_lorda_temp", utilizzazione_completa)
con.execute("""
    CREATE OR REPLACE TABLE presentation.utilizzazione_lorda AS
    SELECT * FROM utilizzazione_lorda_temp
""")

verifica_db = con.execute("""
    SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni, COUNT(DISTINCT anno) AS n_anni
    FROM presentation.utilizzazione_lorda
""").df()
print("\n=== SCRITTURA NEL DATABASE - presentation.utilizzazione_lorda ===")
print(verifica_db.to_string(index=False))

print("\n=== VERIFICA DAL DB - Uta 2025 (valore costante sui 12 mesi) ===")
print(con.execute("""
    SELECT comune, anno, mese, utilizzazione_lorda_pct, segmenti_esclusi
    FROM presentation.utilizzazione_lorda
    WHERE comune = 'Uta' AND anno = 2025
    ORDER BY mese
""").df().to_string(index=False))


=== SCRITTURA NEL DATABASE - presentation.utilizzazione_lorda ===
 n_righe  n_comuni  n_anni
   16224       354       4

=== VERIFICA DAL DB - Uta 2025 (valore costante sui 12 mesi) ===
comune  anno  mese  utilizzazione_lorda_pct  segmenti_esclusi
   Uta  2025     1                     5.84                 1
   Uta  2025     2                     5.84                 1
   Uta  2025     3                     5.84                 1
   Uta  2025     4                     5.84                 1
   Uta  2025     5                     5.84                 1
   Uta  2025     6                     5.84                 1
   Uta  2025     7                     5.84                 1
   Uta  2025     8                     5.84                 1
   Uta  2025     9                     5.84                 1
   Uta  2025    10                     5.84                 1
   Uta  2025    11                     5.84                 1
   Uta  2025    12                     5.84                 1


In [15]:
con.close()
print("Connessione al database chiusa.")

Connessione al database chiusa.
